# 02 — Preprocessing, Applied EDA & Feature Engineering

Covers capstone Step 3: documented cleaning rules, window-disciplined feature
engineering (`src/features.py`), applied EDA, a preliminary embedded feature
selection pass, and dimensionality reduction (PCA scree + UMAP product map).

**Window discipline:** every behavioural aggregate is computed on the feature
window only (2017-01 → 2018-02, `configs/windows.yaml`); the leakage audit cell
asserts it. Customer geography is static (an address is known at serving time)
and is deliberately *not* window-filtered.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.data import load_table
from src.features import (
    assert_max_ts, customer_features, customer_geo, interactions, load_centroids,
    load_clean_items, load_clean_orders, load_clean_products, load_clean_reviews,
    load_windows, pair_features, product_features, sample_negatives, seller_features,
)

sns.set_theme(style="whitegrid")
FIG_DIR = REPO_ROOT / "reports" / "figures"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
SEED = 42

windows = load_windows()
FW, LW = windows["feature_window"], windows["label_window"]

delivered, order_counts = load_clean_orders()
items, item_counts = load_clean_items()
reviews, review_counts = load_clean_reviews()
products, product_counts = load_clean_products()
payments = load_table("order_payments")

for label, c in [("orders", order_counts), ("items", item_counts),
                 ("reviews", review_counts), ("products", product_counts)]:
    print(f"{label:9s} {c}")

orders    {'orders_raw': 99441, 'dropped_not_delivered': 2963}
items     {'item_rows_raw': 112650, 'item_lines_after_unit_dedupe': 102425}
reviews   {'reviews_raw': 99224, 'dropped_duplicate_reviews': 551}
products  {'missing_category': 610, 'imputed_attribute_values': 1838}


## Cleaning rules (each count printed above — nothing dropped silently)

- **Orders:** keep `delivered` only (2,963 dropped; canceled/unavailable orders
  carry no purchase signal and half-shipped ones no delivery outcome).
- **Reviews:** `review_id` is not unique and 547 orders have multiple reviews —
  keep the **latest answer per order** (551 rows dropped).
- **Order items:** unit rows collapsed to one line per (order, product, seller)
  with a quantity column, so interaction matrices count purchases, not units.
- **Products:** missing category → `unknown` then English translation; the seven
  numeric attributes median-imputed (1,838 values across 610+2 products).

In [2]:
# Feature tables (feature window only) + leakage audit
cust = customer_features(delivered, items, payments, reviews, products, FW)
prod = product_features(delivered, items, reviews, products, FW)
sell = seller_features(delivered, items, reviews, FW)
geo = customer_geo()
centroids = load_centroids()

# audit: no feature input postdates the feature-window cutoff
fw_inter = interactions(delivered, items, *FW)
assert_max_ts(fw_inter, "order_purchase_timestamp", FW[1])
print(f"leakage audit passed: max feature-window ts < {FW[1].date()}")

cust.to_csv(PROCESSED_DIR / "customer_features.csv")
prod.to_csv(PROCESSED_DIR / "product_features.csv")
sell.to_csv(PROCESSED_DIR / "seller_features.csv")
print(f"customers with feature-window history: {len(cust)}")
print(f"products: {len(prod)} (has feature-window sales: {prod['has_sales'].mean():.1%})")
print(f"sellers active in feature window: {len(sell)}")
print(f"\ncustomer features:\n{cust.drop(columns=['state', 'zip_code_prefix']).describe().round(2).T.to_string()}")

leakage audit passed: max feature-window ts < 2018-03-01


customers with feature-window history: 55267
products: 32951 (has feature-window sales: 62.0%)
sellers active in feature window: 1945

customer features:
                     count    mean     std   min   25%    50%    75%      max
frequency          55267.0    1.03    0.20  1.00   1.0    1.0    1.0      9.0
monetary           55267.0  139.58  213.02  2.29  47.0   89.0  152.0  13440.0
recency_days       55267.0  155.19  109.88  0.00  63.0  133.0  238.0    419.0
avg_order_value    55267.0  135.67  207.02  2.29  45.9   85.0  149.9  13440.0
median_item_price  55267.0  124.08  185.77  1.20  42.0   79.0  139.9   6735.0
avg_installments   55267.0    2.98    2.75  1.00   1.0    2.0    4.0     24.0
avg_review_given   54863.0    4.13    1.29  1.00   4.0    5.0    5.0      5.0


## Applied EDA

In [3]:
# Delivery time vs review score - Olist's most famous relationship, and the
# domain rationale for the distance/freight features
d = delivered.copy()
d["delivery_days"] = (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.days
dr = d.merge(reviews[["order_id", "review_score"]], on="order_id").dropna(subset=["delivery_days"])

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=dr, x="review_score", y="delivery_days", showfliers=False, ax=ax,
            color="tab:blue")
ax.set_title("Delivery time vs review score")
ax.set_ylabel("days to delivery")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_delivery_vs_review.png", dpi=150)
plt.close(fig)
print(dr.groupby("review_score")["delivery_days"].median().to_string())
late = dr[dr["order_delivered_customer_date"] > dr["order_estimated_delivery_date"]]
print(f"\nlate deliveries: {len(late) / len(dr):.1%} of reviewed orders")
print(f"mean score when late: {late['review_score'].mean():.2f} "
      f"vs on time: {dr.loc[~dr.index.isin(late.index), 'review_score'].mean():.2f}")

review_score
1    16.0
2    13.0
3    12.0
4    10.0
5     9.0

late deliveries: 8.0% of reviewed orders
mean score when late: 2.57 vs on time: 4.29


In [4]:
# Repeat buyers vs one-shot customers (feature-window behaviour)
cmp = cust.assign(repeat=np.where(cust["frequency"] >= 2, "repeat", "one-shot"))
print(
    cmp.groupby("repeat")[
        ["monetary", "avg_order_value", "median_item_price", "avg_review_given", "avg_installments"]
    ].mean().round(2).to_string()
)
print(f"\nrepeat share of feature-window customers: {(cmp['repeat'] == 'repeat').mean():.2%}")

          monetary  avg_order_value  median_item_price  avg_review_given  avg_installments
repeat                                                                                    
one-shot    136.09           136.09             124.63              4.13              2.97
repeat      255.49           121.89             106.08              4.17              3.43

repeat share of feature-window customers: 2.92%


In [5]:
# Category mix by region: is demand geographically uniform? (feeds the fairness
# discussion - a popularity-only recommender inherits the Southeast's taste)
inter_all = interactions(delivered, items, FW[0], windows["holdout"][1])
inter_all = inter_all.merge(products[["product_id", "category"]], on="product_id")
top10 = inter_all["category"].value_counts().head(10).index
pivot = (inter_all[inter_all["category"].isin(top10)]
         .pivot_table(index="category", columns="region", values="order_id", aggfunc="count"))
pivot = pivot / pivot.sum()  # share within region

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set_title("Category share within region (top-10 categories)")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_category_region_heatmap.png", dpi=150)
plt.close(fig)
print("max share deviation across regions per category:")
print((pivot.max(axis=1) - pivot.min(axis=1)).round(3).sort_values(ascending=False).to_string())

max share deviation across regions per category:
category
bed_bath_table           0.106
health_beauty            0.063
telephony                0.062
watches_gifts            0.051
housewares               0.045
furniture_decor          0.042
computers_accessories    0.030
sports_leisure           0.017
auto                     0.014
toys                     0.008


In [6]:
# Winsorisation: price-derived features are capped at the feature-window p99
# (documented threshold) so a handful of luxury items don't stretch every scale
raw_price = prod.loc[prod["has_sales"] == 1, "median_price"]
cap = raw_price.quantile(0.99)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(raw_price, bins=100)
axes[0].set_title(f"product median price (raw, max R${raw_price.max():,.0f})")
axes[1].hist(raw_price.clip(upper=cap), bins=100)
axes[1].set_title(f"winsorised at p99 = R${cap:,.0f}")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_price_winsorisation.png", dpi=150)
plt.close(fig)
print(f"p99 cap: R${cap:.0f}; products affected: {(raw_price > cap).sum()} ({(raw_price > cap).mean():.1%})")

p99 cap: R$1102; products affected: 205 (1.0%)


## Pair assembly & preliminary ranking task

Positives are the label window's delivered (customer, product) purchases;
negatives are uniform-random non-purchased products at 1:4 (seed 42). This is
the **preliminary** protocol — notebook 03 mixes in Stage-1 hard negatives per
the evaluation plan, and re-confirms the feature selection there. Splits group
by customer so the same person never appears on both sides.

In [7]:
pos = interactions(delivered, items, *LW)[["customer_unique_id", "product_id"]].drop_duplicates()
neg = sample_negatives(pos, products["product_id"].to_numpy(), ratio=4, seed=SEED)
pairs = pd.concat([pos.assign(y=1), neg.assign(y=0)], ignore_index=True)
X_all = pair_features(pairs[["customer_unique_id", "product_id"]], cust, geo, prod, sell, centroids)
y = pairs["y"].to_numpy()
groups = pairs["customer_unique_id"].to_numpy()
print(f"pairs: {len(pairs)} ({y.mean():.1%} positive)")
print(f"share of pairs whose customer has feature-window history: {X_all['has_history'].mean():.2%}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(X_all["distance_km"], bins=80)
ax.set_title("Customer -> seller haversine distance (all pairs)")
ax.set_xlabel("km")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_pair_distance.png", dpi=150)
plt.close(fig)
print(X_all["distance_km"].describe().round(1).to_string())

pairs: 106294 (20.0% positive)
share of pairs whose customer has feature-window history: 2.09%
count    106294.0
mean        546.6
std         461.1
min           0.0
25%         363.6
50%         449.3
75%         529.9
max        3843.6


In [8]:
# Correlation structure of the engineered features (numeric only)
num_cols = [c for c in X_all.columns
            if c not in ("customer_unique_id", "product_id", "p_category", "c_region")]
corr = X_all[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax,
            xticklabels=corr.columns, yticklabels=corr.columns)
ax.set_title("Engineered pair-feature correlations")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_corr_heatmap.png", dpi=150)
plt.close(fig)

pairs_hi = (corr.where(np.triu(np.ones_like(corr, dtype=bool), 1)).stack()
            .sort_values(key=np.abs, ascending=False).head(8))
print("strongest correlations:")
print(pairs_hi.round(2).to_string())

strongest correlations:
c_recency_days       has_history           -0.94
c_monetary           c_avg_order_value      0.94
c_frequency          has_history            0.86
c_avg_order_value    c_median_item_price    0.85
c_frequency          c_recency_days        -0.83
c_monetary           c_median_item_price    0.80
c_avg_installments   has_history            0.75
c_median_item_price  has_history            0.74


## Encoding & scaling

Region (5 values + unknown) is one-hot encoded; product category (71 values) is
frequency-encoded with its feature-window order share, which avoids a 71-column
one-hot the trees don't need. Linear models get a `StandardScaler` inside their
pipeline; tree models are left unscaled — splits are invariant to monotone
transforms, so scaling trees would add a step without changing anything.

In [9]:
cat_share = (fw_inter.merge(products[["product_id", "category"]], on="product_id")
             ["category"].value_counts(normalize=True))
X = X_all[num_cols].copy()
X["p_category_freq"] = X_all["p_category"].map(cat_share).fillna(0.0)
X = pd.concat([X, pd.get_dummies(X_all["c_region"], prefix="region", dtype=float)], axis=1)
feature_names = list(X.columns)
print(f"model matrix: {X.shape}")

from sklearn.model_selection import GroupShuffleSplit

tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
                      .split(X, y, groups))
X_tr, X_va, y_tr, y_va = X.iloc[tr_idx], X.iloc[va_idx], y[tr_idx], y[va_idx]
print(f"train {len(X_tr)}, val {len(X_va)} (grouped by customer)")

model matrix: (106294, 30)


train 84921, val 21373 (grouped by customer)


In [10]:
# Embedded feature selection 1: L1-regularised logistic regression path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

l1_results = {}
for C in (0.001, 0.01, 0.1):
    lr = make_pipeline(StandardScaler(),
                       LogisticRegression(l1_ratio=1, solver="liblinear", C=C,
                                          max_iter=2000, random_state=SEED))
    lr.fit(X_tr, y_tr)
    coefs = lr.named_steps["logisticregression"].coef_[0]
    auc = roc_auc_score(y_va, lr.predict_proba(X_va)[:, 1])
    l1_results[C] = (coefs, auc)
    print(f"C={C:<6} zeroed {int((coefs == 0).sum()):2d}/{len(coefs)} features, val AUC {auc:.4f}")

C_SEL = 0.01  # keeps AUC while zeroing the dead weight
sel_coefs, _ = l1_results[C_SEL]
l1_selected = [f for f, c in zip(feature_names, sel_coefs) if c != 0]
print(f"\nL1-selected at C={C_SEL} ({len(l1_selected)} features):")
order = np.argsort(-np.abs(sel_coefs))
for i in order[:12]:
    if sel_coefs[i] != 0:
        print(f"  {feature_names[i]:32s} {sel_coefs[i]:+.3f}")

C=0.001  zeroed 26/30 features, val AUC 0.7334

C=0.01   zeroed 12/30 features, val AUC 0.7253


C=0.1    zeroed  3/30 features, val AUC 0.7237

L1-selected at C=0.01 (18 features):
  p_popularity                     +1.860
  p_has_sales                      -0.364
  category_match                   +0.092
  s_seller_order_count             +0.086
  same_state                       +0.059
  p_median_price_w                 -0.058
  p_product_description_lenght     +0.056
  s_seller_review_mean             -0.051
  has_history                      -0.025
  p_freight_ratio                  -0.019
  p_review_mean                    -0.008
  c_recency_days                   +0.006


In [11]:
# Embedded feature selection 2: XGBoost gain importance
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.9,
                    tree_method="hist", random_state=SEED, n_jobs=-1,
                    eval_metric="auc")
xgb.fit(X_tr, y_tr)
auc_xgb_full = roc_auc_score(y_va, xgb.predict_proba(X_va)[:, 1])
gain = pd.Series(xgb.feature_importances_, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 7))
sns.barplot(x=gain.values, y=gain.index, ax=ax, color="tab:blue")
ax.set_title(f"XGBoost gain importance (val AUC {auc_xgb_full:.4f})")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_xgb_importance.png", dpi=150)
plt.close(fig)
print(gain.head(12).round(4).to_string())

p_popularity                    0.4059
category_match                  0.0578
s_seller_order_count            0.0368
p_review_mean                   0.0308
s_seller_review_mean            0.0302
p_price_band                    0.0278
p_category_satisfaction_rate    0.0276
p_product_weight_g              0.0269
p_category_freq                 0.0263
p_median_price_w                0.0262
p_product_description_lenght    0.0261
p_product_photos_qty            0.0251


In [12]:
# Before/after: keep the union of L1-survivors and top-gain features, drop the rest
keep = sorted(set(l1_selected) | set(gain.head(15).index))
dropped = [f for f in feature_names if f not in keep]
print(f"kept {len(keep)}, dropped {len(dropped)}: {dropped}")

lr_full = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
lr_full.fit(X_tr, y_tr)
auc_lr_full = roc_auc_score(y_va, lr_full.predict_proba(X_va)[:, 1])

lr_red = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
lr_red.fit(X_tr[keep], y_tr)
auc_lr_red = roc_auc_score(y_va, lr_red.predict_proba(X_va[keep])[:, 1])

xgb_red = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.9,
                        tree_method="hist", random_state=SEED, n_jobs=-1, eval_metric="auc")
xgb_red.fit(X_tr[keep], y_tr)
auc_xgb_red = roc_auc_score(y_va, xgb_red.predict_proba(X_va[keep])[:, 1])

print(f"\nLogisticRegression  AUC: full {auc_lr_full:.4f} -> reduced {auc_lr_red:.4f}")
print(f"XGBoost             AUC: full {auc_xgb_full:.4f} -> reduced {auc_xgb_red:.4f}")
print("(preliminary - random negatives only; notebook 03 re-runs selection on the real protocol)")

kept 21, dropped 9: ['c_frequency', 'c_monetary', 'c_avg_order_value', 'c_median_item_price', 'c_avg_installments', 'c_avg_review_given', 'distance_km', 'region_North', 'region_Southeast']



LogisticRegression  AUC: full 0.7234 -> reduced 0.7237
XGBoost             AUC: full 0.8134 -> reduced 0.8157
(preliminary - random negatives only; notebook 03 re-runs selection on the real protocol)


## Dimensionality reduction

PCA quantifies the redundancy in the product feature space (the scree plot also
justifies how many components carry signal); UMAP gives the 2-D product map,
coloured by category, to eyeball whether content features separate categories —
that's the machinery the content-based candidate generator relies on.

In [13]:
from sklearn.decomposition import PCA

prod_num_cols = ["popularity", "median_price_w", "freight_ratio", "review_mean",
                 "category_satisfaction_rate", "has_sales", "product_photos_qty",
                 "product_name_lenght", "product_description_lenght", "product_weight_g",
                 "product_length_cm", "product_height_cm", "product_width_cm"]
P = StandardScaler().fit_transform(prod[prod_num_cols])
pca = PCA(random_state=SEED).fit(P)
cum = np.cumsum(pca.explained_variance_ratio_)
n90 = int(np.argmax(cum >= 0.90) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(cum) + 1), pca.explained_variance_ratio_, label="per component")
ax.plot(range(1, len(cum) + 1), cum, marker="o", color="tab:red", label="cumulative")
ax.axhline(0.9, ls="--", c="grey")
ax.set_title(f"PCA on product features - {n90} of {len(prod_num_cols)} components reach 90%")
ax.set_xlabel("component")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "02_pca_scree.png", dpi=150)
plt.close(fig)
print(f"components for 90% variance: {n90} of {len(prod_num_cols)}")
print("cumulative:", np.round(cum, 3))

components for 90% variance: 10 of 13


cumulative: [0.198 0.305 0.403 0.494 0.573 0.648 0.716 0.781 0.843 0.901 0.941 0.975
 1.   ]


In [14]:
import umap

rng = np.random.default_rng(SEED)
top8 = prod["category"].value_counts().head(8).index
sample_idx = rng.choice(len(prod), size=8000, replace=False)
emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(P[sample_idx])
cat_sample = prod["category"].to_numpy()[sample_idx]

fig, ax = plt.subplots(figsize=(9, 7))
other = ~pd.Series(cat_sample).isin(top8).to_numpy()
ax.scatter(emb[other, 0], emb[other, 1], s=3, c="lightgrey", label="other")
for cat in top8:
    m = cat_sample == cat
    ax.scatter(emb[m, 0], emb[m, 1], s=4, label=cat)
ax.legend(markerscale=3, fontsize=8, loc="best")
ax.set_title("UMAP of product feature space (8,000-product sample)")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_product_umap.png", dpi=150)
plt.close(fig)
print("UMAP done")

C:\Users\jmonz\Documents\GitHub\ecommerce-recommender-capstone\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\jmonz\Documents\GitHub\ecommerce-recommender-capstone\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP done


## Takeaways

- **Cleaning cost almost nothing and hid nothing:** 2,963 non-delivered orders
  dropped (of 99,441), 551 duplicate reviews removed, 112,650 unit rows collapsed
  to 102,425 order lines, and 1,838 product-attribute values median-imputed - every
  count printed above.
- **Delivery experience is the satisfaction story.** Median delivery is 9 days on
  5-star orders vs 16 days on 1-star, and the 8% of orders that arrive late average
  2.57 stars against 4.29 on time. That's the domain justification for the freight
  and distance features.
- **Cold start dominates the ranking task:** only 2.09% of label-window pairs belong
  to customers with feature-window history. The preliminary selection dropped all six
  customer behavioural features, and product popularity towers over everything
  (XGBoost gain 0.41 vs 0.06 for the runner-up).
- **Selection sanity check passed:** removing the 9 dropped features left logistic
  regression flat (0.7234 -> 0.7237 AUC) and nudged XGBoost up (0.8134 -> 0.8157).
  With uniform-random negatives this task is popularity-easy; notebook 03 re-runs
  the selection with Stage-1 hard negatives before anything is final.
- **PCA says the product features aren't very redundant** (10 of 13 components for
  90% variance), while the UMAP map still shows clear category structure - which is
  what the content-based candidate generator needs.
- The near-perfect correlations in the heatmap (e.g. recency x has_history at -0.94)
  are structural artefacts of the cold-start fills, worth remembering when reading
  feature importances and SHAP values later.